# Phase 2, Part 1: Preparing the Accident Tables

This project focuses on construction safety, so construction accidents are the cases I want to predict well. But OSHA investigates accidents in every industry, and falls, struck by events and caught in events behave similarly across industries. That means accident records from other industries might still help a model learn.

So I prepare one table that holds every labeled accident, with flags that mark which ones are construction. This lets me run a fair experiment later: train on construction only, train on all industries, and score both on construction cases only.

In this notebook I:

1. Load every labeled injury record from the full OSHA enforcement data
2. Link each one to its inspection and flag construction with two definitions, a strict one and a broad one
3. Attach and clean the narrative text, removing words that reveal the outcome
4. Split the columns into circumstances (what was happening) and injury details (what the injury was)
5. Test each column for leakage by its values, not only by whether it is filled in
6. Save one table for the baseline models

In [1]:
from pathlib import Path
import re
import sqlite3
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupKFold, cross_val_score

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 100)

RANDOM_STATE = 42

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
DB_PATH = RAW / "osha_enforcement.db"

print("Project root:", ROOT)
print("Database found:", DB_PATH.exists())
conn = sqlite3.connect(str(DB_PATH))

Project root: c:\Users\sabin\constructiq
Database found: True


## 1. Loading accidents and injuries

I load the accident and injury tables. The severity label is `degree_of_inj`, where 1 is a fatality, 2 is hospitalized and 3 is not hospitalized. A code of 0 and a few missing values have no usable label, so I leave them out.

In [2]:
accident = pd.read_sql("SELECT * FROM accident", conn)
injury = pd.read_sql("SELECT * FROM accident_injury", conn)

for d in (accident, injury):
    d["summary_nr"] = pd.to_numeric(d["summary_nr"], errors="coerce").astype("Int64")

injury["rel_insp_nr"] = pd.to_numeric(injury["rel_insp_nr"], errors="coerce").astype("Int64")
injury["degree_of_inj"] = pd.to_numeric(injury["degree_of_inj"], errors="coerce")

print("Injury rows before filtering:", len(injury))
injury = injury[injury["degree_of_inj"].isin([1, 2, 3])].copy()
print("Injury rows with a usable severity label:", len(injury))

Injury rows before filtering: 231809
Injury rows with a usable severity label: 226291


## 2. Linking to inspections and flagging construction

Each injury links to an inspection, and the inspection holds the industry codes. I flag construction in two ways:

- **Strict:** the NAICS code starts with 23, or the older SIC code starts with 15, 16 or 17. These are the official construction industry codes.
- **Broad:** the strict rule, or the inspection cited at least one standard from the 1926 construction series.

The strict version is my main definition. The broad version adds more cases but includes some that may not be construction, so I use it as a robustness check.

In [3]:
wanted = set(injury["rel_insp_nr"].dropna().astype("int64"))

cols = "activity_nr, naics_code, sic_code, site_state, open_date"
parts = []
for chunk in pd.read_sql(f"SELECT {cols} FROM inspection", conn, chunksize=500_000):
    chunk["activity_nr"] = pd.to_numeric(chunk["activity_nr"], errors="coerce")
    parts.append(chunk[chunk["activity_nr"].isin(wanted)])

insp = pd.concat(parts, ignore_index=True).drop_duplicates("activity_nr")
insp["activity_nr"] = insp["activity_nr"].astype("int64")
print("Inspections found:", len(insp), "of", len(wanted))

Inspections found: 188584 of 188652


In [4]:
# Inspections that cited at least one construction standard 
cited = pd.read_sql("SELECT DISTINCT activity_nr FROM violation WHERE TRIM(standard) LIKE '1926%'", conn)
cited_set = set(pd.to_numeric(cited["activity_nr"], errors="coerce").dropna().astype("int64"))
print("Inspections that cited a 1926 standard:", len(cited_set))

Inspections that cited a 1926 standard: 1170763


In [5]:
acc_cols = ["summary_nr", "event_date", "event_desc", "event_keyword", "const_end_use",
            "build_stories", "nonbuild_ht", "project_cost", "project_type"]

full = injury.merge(insp, left_on="rel_insp_nr", right_on="activity_nr", how="left")
full = full.merge(accident[acc_cols], on="summary_nr", how="left")

naics = full["naics_code"].astype("string").str.strip()
sic = full["sic_code"].astype("string").str.strip()

full["rule_naics"] = naics.str.startswith("23").fillna(False).astype(bool)
full["rule_sic"] = sic.str[:2].isin(["15", "16", "17"]).fillna(False).astype(bool)
full["rule_cited_1926"] = full["rel_insp_nr"].isin(cited_set).fillna(False).astype(bool)

full["is_construction_strict"] = full["rule_naics"] | full["rule_sic"]
full["is_construction_broad"] = full["is_construction_strict"] | full["rule_cited_1926"]

label_map = {1: "Fatality", 2: "Hospitalized", 3: "Not hospitalized"}
for name, mask in [
    ("All industries", full["summary_nr"].notna()),
    ("Construction, strict", full["is_construction_strict"]),
    ("Construction, broad", full["is_construction_broad"]),
    ("Not construction (strict)", ~full["is_construction_strict"]),
]:
    sub = full[mask]
    print(name)
    print("  rows:", len(sub), "| unique accidents:", sub["summary_nr"].nunique())
    print("  ", sub["degree_of_inj"].map(label_map).value_counts().to_dict())
    print()

All industries
  rows: 226291 | unique accidents: 187074
   {'Hospitalized': 99885, 'Fatality': 93068, 'Not hospitalized': 33338}

Construction, strict
  rows: 70979 | unique accidents: 60292
   {'Fatality': 33659, 'Hospitalized': 30415, 'Not hospitalized': 6905}

Construction, broad
  rows: 74523 | unique accidents: 63180
   {'Fatality': 35882, 'Hospitalized': 31437, 'Not hospitalized': 7204}

Not construction (strict)
  rows: 155312 | unique accidents: 127149
   {'Hospitalized': 69470, 'Fatality': 59409, 'Not hospitalized': 26433}



## 3. Attaching the narrative text

Each accident has a narrative stored as several lines of fixed width, 80 characters each. The lines cut words in the middle, for example "Th" at the end of one line and "ey" at the start of the next. Some lines also lose their trailing space when they are stored, which glues two words together. Before joining, I pad every line back to its full width of 80 characters, which restores the spaces lost at line breaks. I then join the lines end to end with no added separator and collapse repeated spaces. As a check, I count narratives that still contain glued words such as "waskilled". Only a very small number remain, under half a percent.

In [6]:
ids = set(full["summary_nr"].dropna().astype("int64"))

parts = []
for chunk in pd.read_sql("SELECT summary_nr, line_nr, abstract_text FROM accident_abstract", conn, chunksize=500_000):
    chunk["summary_nr"] = pd.to_numeric(chunk["summary_nr"], errors="coerce")
    parts.append(chunk[chunk["summary_nr"].isin(ids)])

abstract = pd.concat(parts, ignore_index=True).dropna(subset=["summary_nr"])
abstract["summary_nr"] = abstract["summary_nr"].astype("int64")
abstract["line_nr"] = pd.to_numeric(abstract["line_nr"], errors="coerce")

narratives = (
    abstract.sort_values(["summary_nr", "line_nr"])
    .groupby("summary_nr")["abstract_text"]
    .apply(lambda lines: "".join(str(l).ljust(80) for l in lines if pd.notna(l)))
    .reset_index()
    .rename(columns={"abstract_text": "full_narrative"})
)
narratives["summary_nr"] = narratives["summary_nr"].astype("Int64")
narratives["full_narrative"] = narratives["full_narrative"].str.replace(r"\s+", " ", regex=True).str.strip()

full = full.merge(narratives, on="summary_nr", how="left")
print("Rows:", len(full))
print("Rows with a narrative:", int(full["full_narrative"].notna().sum()))

glued = r"\b(?:was|and|not|without|were|had)(?:killed|hospitalized|treated|admitted|hospitalization|taken|transported)\b"
print("Narratives with glued words:", int(full["full_narrative"].fillna("").str.lower().str.contains(glued, regex=True).sum()))

Rows: 226291
Rows with a narrative: 192166
Narratives with glued words: 692


## 4. Cleaning the text

Narratives are written after the event, so they often describe the outcome. A text model would then learn the words killed, died and hospital instead of learning about risk. I remove outcome words and hospital words from the text for every class, so the model has to learn from what happened, not how it ended.

In [7]:
OUTCOME_WORDS = (
    r"\b(?:died|dies|die|dead|death|deaths|killed|kills|kill|fatal|fatally|fatality|fatalities|"
    r"deceased|expired|pronounced|perished|succumbed|hospital|hospitals|hospitalized|hospitalised|"
    r"admitted|emergency room|ambulance|transported|airlifted|life flight|intensive care|icu|coma|surgery)\b"
)

raw_text = (full["event_desc"].fillna("") + " " + full["event_keyword"].fillna("") + " " +
            full["full_narrative"].fillna("")).str.lower()

before = raw_text.str.contains(OUTCOME_WORDS, regex=True)
print("Share of texts with outcome or hospital words BEFORE cleaning, by severity:")
print(before.groupby(full["degree_of_inj"].map(label_map)).mean().round(3))

full["text_clean"] = raw_text.str.replace(OUTCOME_WORDS, " ", regex=True)
after = full["text_clean"].str.contains(OUTCOME_WORDS, regex=True)
print()
print("Share still containing them AFTER cleaning:", round(after.mean(), 3))

Share of texts with outcome or hospital words BEFORE cleaning, by severity:
degree_of_inj
Fatality            0.733
Hospitalized        0.713
Not hospitalized    0.508
dtype: float64

Share still containing them AFTER cleaning: 0.0


## 5. Two groups of columns

I split the structured columns into two groups. **Circumstances** describe what was happening when the accident occurred, such as the type of event, the source, the human factor, the task and the age of the worker. **Injury details** describe the result, such as the nature of the injury and the body part. Injury details are only known after the accident, so my main models will use circumstances only. The injury details are an optional second set.

The age field uses 0 when the age is unknown, so I treat 0 as missing.

In [8]:
CODED = ["age", "nature_of_inj", "part_of_body", "src_of_injury", "event_type", "evn_factor", "hum_factor",
         "occ_code", "task_assigned", "const_op", "const_op_cause", "fat_cause", "build_stories", "nonbuild_ht"]
CATEG = ["const_end_use", "project_cost", "project_type"]

for c in CODED:
    full[c] = pd.to_numeric(full[c], errors="coerce")
full["age"] = full["age"].where(full["age"] > 0)

CIRCUMSTANCE_COLS = ["age", "event_type", "src_of_injury", "evn_factor", "hum_factor", "task_assigned",
                     "occ_code", "const_op", "const_op_cause", "build_stories", "nonbuild_ht"] + CATEG
INJURY_COLS = ["nature_of_inj", "part_of_body", "fat_cause"]

full["event_date"] = pd.to_datetime(full["event_date"], errors="coerce", utc=True)
full["is_fatal"] = (full["degree_of_inj"] == 1).astype(int)
full["severity_3class"] = full["degree_of_inj"].astype(int)
full = full.reset_index(drop=True)

print("Circumstance columns:", CIRCUMSTANCE_COLS)
print("Injury detail columns:", INJURY_COLS)

Circumstance columns: ['age', 'event_type', 'src_of_injury', 'evn_factor', 'hum_factor', 'task_assigned', 'occ_code', 'const_op', 'const_op_cause', 'build_stories', 'nonbuild_ht', 'const_end_use', 'project_cost', 'project_type']
Injury detail columns: ['nature_of_inj', 'part_of_body', 'fat_cause']


## 6. Testing each column for leakage by its values

My earlier check only asked whether a column is filled in. Here I ask a stronger question: how well can a very small decision tree predict fatal versus non fatal using only that one column, tested on accidents it has not seen? One column should not predict the outcome far better than guessing. If it does, it is leaking, or it is so tied to the outcome that I should not call it a prediction.

In [9]:
strict = full[full["is_construction_strict"]].reset_index(drop=True)
majority = max(strict["is_fatal"].mean(), 1 - strict["is_fatal"].mean())

def encode(s):
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(-1).to_numpy().reshape(-1, 1)
    return pd.factorize(s.astype("string").fillna("missing"))[0].reshape(-1, 1)

groups = strict["summary_nr"].astype("int64")
rows = []
for c in CIRCUMSTANCE_COLS + INJURY_COLS:
    scores = cross_val_score(
        DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, random_state=RANDOM_STATE),
        encode(strict[c]), strict["is_fatal"], groups=groups,
        cv=GroupKFold(n_splits=5), scoring="accuracy",
    )
    rows.append((c, "circumstance" if c in CIRCUMSTANCE_COLS else "injury detail", round(scores.mean(), 3)))

single = pd.DataFrame(rows, columns=["column", "group", "accuracy_alone"]).sort_values("accuracy_alone", ascending=False)
print("Majority class accuracy:", round(majority, 3))
print(single.to_string(index=False))

Majority class accuracy: 0.526
        column         group  accuracy_alone
  part_of_body injury detail           0.691
 nature_of_inj injury detail           0.655
      occ_code  circumstance           0.617
 src_of_injury  circumstance           0.592
           age  circumstance           0.582
    event_type  circumstance           0.581
     fat_cause injury detail           0.561
 build_stories  circumstance           0.556
 const_end_use  circumstance           0.554
    evn_factor  circumstance           0.553
 task_assigned  circumstance           0.549
    hum_factor  circumstance           0.543
  project_type  circumstance           0.543
const_op_cause  circumstance           0.541
      const_op  circumstance           0.526
  project_cost  circumstance           0.526
   nonbuild_ht  circumstance           0.526


## 7. Saving the table

I save one table with every labeled accident, the construction flags, the cleaned text, both column groups and the targets. The rule columns are kept for reference only and will not be used as model features.

In [10]:
full = full.rename(columns={"naics_code": "audit_naics_code", "site_state": "audit_state", "open_date": "audit_open_date"})
for r in ["rule_naics", "rule_sic", "rule_cited_1926"]:
    full[f"audit_{r}"] = full[r].astype(int)

keep = (["summary_nr", "event_date", "degree_of_inj", "severity_3class", "is_fatal",
         "is_construction_strict", "is_construction_broad", "text_clean"]
        + CIRCUMSTANCE_COLS + INJURY_COLS
        + ["audit_naics_code", "audit_state", "audit_open_date", "audit_rule_naics", "audit_rule_sic", "audit_rule_cited_1926"])

table = full[keep].sort_values("event_date").reset_index(drop=True)
out_path = PROC / "accidents_labeled_all_industries.csv"
table.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Rows:", len(table), "| unique accidents:", table["summary_nr"].nunique())
print("File size (MB):", round(out_path.stat().st_size / 1e6, 1))

Saved: c:\Users\sabin\constructiq\data\processed\accidents_labeled_all_industries.csv
Rows: 226291 | unique accidents: 187074
File size (MB): 157.4


In [11]:
# How do the classes change over time in the strict construction group?
s = table[table["is_construction_strict"]].copy()
s["decade"] = (s["event_date"].dt.year // 10 * 10).astype("Int64")
print(pd.crosstab(s["decade"], s["severity_3class"].map(label_map), normalize="index").round(3))
print()
print(pd.crosstab(s["decade"], s["severity_3class"].map(label_map)))

severity_3class  Fatality  Hospitalized  Not hospitalized
decade                                                   
1970                0.000         0.000             1.000
1980                0.529         0.349             0.121
1990                0.461         0.427             0.112
2000                0.424         0.473             0.102
2010                0.391         0.532             0.077
2020                0.588         0.368             0.044

severity_3class  Fatality  Hospitalized  Not hospitalized
decade                                                   
1970                    0             0                 1
1980                 4140          2731               948
1990                 6910          6394              1684
2000                 7813          8718              1884
2010                 5648          7677              1118
2020                 3011          1885               224


In [12]:
conn.close()

## Where this leaves me

I now have one table that holds every labeled OSHA accident, with two definitions of construction and a clean set of features. In the next notebook I use it to train the baseline models. The main experiment compares three setups: construction only, all industries, and the broad construction definition, always scored on strict construction cases. I report every result with its spread.